In [25]:
import os

In [26]:
app_src_dir = "./app"
os.makedirs(app_src_dir, exist_ok=True)

In [ ]:
%%writefile {app_src_dir}/log_in.py
import streamlit as st

import streamlit as st
from azure.storage.blob import BlobServiceClient
import utils

connection_string = st.secrets["AZURE_STORAGE_CONNECTION_STRING"]
container_name = "recsys"
user_map = "user_map/user_map.csv"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

user_map_df = utils.blob_to_df(container_client, user_map)
name_to_id = dict(zip(user_map_df["name"], user_map_df["user_id"]))

st.title("Log in")
name_input = st.text_input("Enter your name to continue:")

if st.button("Log In"):
    if name_input in name_to_id:
        user_id = name_to_id[name_input]
        st.session_state.logged_in_user = user_id
        st.success(f"Logged in as {name_input} (ID: {user_id})")
        st.switch_page("recommendations.py")
    else:
        st.error("Name not recognized. Please try again.")

In [42]:
%%writefile {app_src_dir}/recommendations.py
import streamlit as st
from azure.storage.blob import BlobServiceClient
import utils

st.set_page_config(
page_title="Recommender App",
)

connection_string = st.secrets["AZURE_STORAGE_CONNECTION_STRING"]
container_name = "recsys"
ranked_recommendations = "ranked_recommendations/ranked_recommendations.csv"
raw_interactions = "interactions/interactions.csv"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

ranked_recommendations_df = utils.blob_to_df(container_client, ranked_recommendations)
ranked_recommendations_df = ranked_recommendations_df.reset_index(drop=True)
raw_interactions_df = utils.blob_to_df(container_client, raw_interactions)

# Declare session state for when like button is clicked
if 'likes_clicked' not in st.session_state:
    st.session_state.likes_clicked = {}

# Declare session state for when dislike button is clicked
if 'dislikes_clicked' not in st.session_state:
    st.session_state.dislikes_clicked = {}

# Declare session state for when shares button is clicked
if 'shares_clicked' not in st.session_state:
    st.session_state.shares_clicked = {}

# Declare session state for when report button is clicked
if 'reports_clicked' not in st.session_state:
    st.session_state.reports_clicked = {}

if "news_clicked" not in st.session_state:
    st.session_state.news_clicked = False

st.title('Recommended News')

if "logged_in_user" not in st.session_state:
    st.warning("Please log in first.")
    st.stop()

selected_user = st.session_state.logged_in_user

st.divider() 

rec_index = 0
utils.show_recommendations(
    ranked_recommendations_df, 
    selected_user, 
    rec_index, 
    rec_index+100,
    raw_interactions_df,
    container_client, 
    raw_interactions)

Overwriting ./app/recommendations.py


In [27]:
%%writefile {app_src_dir}/manage_account.py
import streamlit as st 
import pandas as pd
import numpy as np
from azure.storage.blob import BlobServiceClient
import utils

connection_string = st.secrets["AZURE_STORAGE_CONNECTION_STRING"]
container_name = "recsys"
raw_content = "content/content.csv"
raw_users = "users/users.csv"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

st.title('Manage your preference')

st.header('What content do you want to see the most?')

if "logged_in_user" not in st.session_state:
    st.warning("Please log in first.")
    st.stop()

raw_users_df = utils.blob_to_df(container_client, raw_users)

content_df = utils.blob_to_df(container_client, raw_content)
categories = list(content_df['category'].unique())
sub_categories = list(content_df['sub_category'])

selection = st.pills("News Category - select multiple", categories, selection_mode="multi")

user_selection = pd.DataFrame({
    "user_id":[st.session_state.logged_in_user],
    "liked_cat": [selection]
    }
)

if selection:
    filtered = [item for item in sub_categories if any(word in item for word in selection)]
    cleaned = list(
        set(
            cleaned for item in filtered
            if (cleaned := utils.remove_prefixes(item, selection).strip())
            )
        )
    sub_selection = st.pills('Sub Category - select multiple', cleaned, selection_mode='multi')

    if sub_selection:
        user_selection['liked_subcat'] = [sub_selection]

raw_users_df = raw_users_df.merge(user_selection, on='user_id', how='left', suffixes=('', '_new'))

if 'liked_cat_new' in raw_users_df.columns:
    raw_users_df['liked_cat_new'] = np.where(
        raw_users_df['liked_cat_new'].notna(),
        raw_users_df['liked_cat_new'],
        raw_users_df['liked_cat']
    )
    raw_users_df.drop(columns='liked_cat', inplace=True)
    raw_users_df = raw_users_df.rename(columns={'liked_cat_new': 'liked_cat'})

if 'liked_subcat_new' in raw_users_df.columns:
    raw_users_df['liked_subcat_new'] = np.where(
        raw_users_df['liked_subcat_new'].notna(),
        raw_users_df['liked_subcat_new'],
        raw_users_df['liked_subcat']
    )
    raw_users_df.drop(columns='liked_subcat', inplace=True)
    raw_users_df = raw_users_df.rename(columns={'liked_subcat_new': 'liked_subcat'})

if 'liked_cat' in raw_users_df.columns:
    merged_users_df = raw_users_df.groupby('user_id', as_index=False).agg({
        'is_child': 'first',
        'name': 'first',
        'restricted_tags': 'first',
        'liked_cat': list,
    })

if 'liked_subcat' in raw_users_df.columns:
    merged_users_df = raw_users_df.groupby('user_id', as_index=False).agg({
        'is_child': 'first',
        'name': 'first',
        'restricted_tags': 'first',
        'liked_cat': list,
        'liked_subcat': list,
    })
#print(merged_users_df[merged_users_df['user_id']==st.session_state.logged_in_user])

container_client.get_blob_client(raw_users).upload_blob(
        merged_users_df.to_csv(index=False), overwrite=True)

Overwriting ./app/manage_account.py


In [29]:
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from io import BytesIO
import pandas as pd

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "recsys"
raw_content = "content/content.csv"
raw_users = "users/users.csv"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_name)

def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

raw_users_df = blob_to_df(container_client, raw_users)
raw_users_df[raw_users_df['user_id']=='U19519']

#raw_users_df = raw_users_df.drop_duplicates()
#raw_users_df = raw_users_df.drop(columns=['liked_cat_new'])

#container_client.get_blob_client(raw_users).upload_blob(
#        raw_users_df.to_csv(index=False), overwrite=True)

,user_id,is_child,name,restricted_tags,liked_cat,liked_subcat
11,U19519,False,Sandy Gonzalez,NaN,"[['Lifestyle', 'Music']]","[['Pets', 'DIY']]"


In [5]:
%%writefile {app_src_dir}/learn.py
import streamlit as st 
st.title('Learn about us')

st.header("✨ How to Use This App")

st.markdown("""
Welcome! 👋 This simple app gives you personalised news recommendations.

### 📋 Steps to Get Started:
1. 🔐 **Please log in** with your **name** on the login page.
2. 📊 You will then see **recommendations** tailored just for you.
3. 💬 **Interact** with the content — like, dislike, or explore — just like you would on a social media app!

""")

st.header('❓ What is this app for?')
st.markdown("""
The aim of this project is to produce recommende system through which we can do the following:

1. produce recommendations from established interactions
2. show new recommendations and allow users to create new interactions which can be used to update future recommendations
3. extract safety metrics from held data on recommendations and interactions
""")

st.write('This app is developed by Qiq and Torid 💗')

Overwriting ./app/learn.py


In [ ]:
%%writefile {app_src_dir}/app.py
import streamlit as st 
pages = {
    "Your account": [
        st.Page("log_in.py", title="Log in"),
        st.Page("manage_account.py", title='Manage your account')
    ],
    "Recommendations": [
        st.Page("recommendations.py", title="Your news recommendations")
    ],
    "Resources": [
        st.Page("learn.py", title="Learn about us")
    ]
}

pg = st.navigation(pages)
pg.run()

In [40]:
%%writefile {app_src_dir}/utils.py
import streamlit as st
import pandas as pd
import pickle
import os
import time
from io import BytesIO
import random
from datetime import datetime

def remove_prefixes(text, prefixes):
    for prefix in prefixes:
        if text.startswith(prefix):
            return text[len(prefix):]
    return text

def blob_to_df(container_client, blob_name):
    client = container_client.get_blob_client(blob_name)
    data = client.download_blob().readall()
    stream = BytesIO(data)
    df = pd.read_csv(stream)
    return df

@st.cache_data
def state_to_df(types='like'):
    if types == 'like':
        df = pd.DataFrame(st.session_state.likes_clicked.items(), columns=['content_id', "Like"])
        df['content_id'] = df['content_id'].str.replace('like_count_', '')   
    elif types == 'dislike':
        df = pd.DataFrame(st.session_state.dislikes_clicked.items(), columns=['content_id', "Dislike"])
        df['content_id'] = df['content_id'].str.replace('dislike_count_', '')   
    elif types == 'share':
        df = pd.DataFrame(st.session_state.shares_clicked.items(), columns=['content_id', "Share"])
        df['content_id'] = df['content_id'].str.replace('share_count_', '')   
    elif types == 'report':
        df = pd.DataFrame(st.session_state.reports_clicked.items(), columns=['content_id', "Report"])
        df['content_id'] = df['content_id'].str.replace('report_count_', '')   

    return df 

def show_recommendations(recs, user, n, n1, int_df, container_client, int_client):

    #selected_user1 = 'U80234' # for testing
    recommendations = recs[recs['user_id'] == user].iloc[n:n1, :]
    print(len(recommendations))

    for i in range(len(recommendations)):
        ids = recommendations.iloc[i]['content_id']
        title = recommendations.iloc[i]['title']
        category = recommendations.iloc[i]['category']
        subCategory = recommendations.iloc[i]['sub_category']
        abstract = recommendations.iloc[i]['abstract']

        container = st.container(border=True)

        with container:
            st.subheader(title)
            st.write(f'{category} | {subCategory}')

            col11, col12 = st.columns(2)

            with col11:
                st.image('imgs/news_image.jpg')
            
            with col12: 
                st.markdown(abstract)

            with st.expander('View Full Story'):
                st.write('Full Story')
                # TODO: add full story here

            col21, col22, col23, col24 = st.columns(4)

            with col21:

                like_key = f"like_clicked_{ids}"  
                like_count_key = f"like_count_{ids}"

                if like_key not in st.session_state:
                    st.session_state[like_key] = False  

                if like_count_key not in st.session_state.likes_clicked:
                    st.session_state.likes_clicked[like_count_key] = 0 

                def handle_like_button(like_key, like_count_key):
                    st.session_state[like_key] = True
                    st.session_state.likes_clicked[like_count_key] += 1

                if not st.session_state[like_key]:
                    st.button(
                        f"👍 {st.session_state.likes_clicked[like_count_key]}",
                        key=f"like_button_{ids}",
                        on_click=handle_like_button,
                        args=(like_key, like_count_key),
                    )
                else:
                    st.button(
                        f"👍 {st.session_state.likes_clicked[like_count_key]}",
                        key=f"like_button_{ids}",
                        disabled=True 
                    )

            with col22:

                dislike_key = f"dislike_clicked_{ids}"  
                dislike_count_key = f"dislike_count_{ids}"

                if dislike_key not in st.session_state:
                    st.session_state[dislike_key] = False  

                if dislike_count_key not in st.session_state.dislikes_clicked:
                    st.session_state.dislikes_clicked[dislike_count_key] = 0 

                def handle_dislike_button(dislike_key, dislike_count_key):
                    st.session_state[dislike_key] = True
                    st.session_state.dislikes_clicked[dislike_count_key] += 1

                if not st.session_state[dislike_key]:
                    st.button(
                        f"👎 {st.session_state.dislikes_clicked[dislike_count_key]}",
                        key=f"dislike_button_{ids}",
                        on_click=handle_dislike_button,
                        args=(dislike_key, dislike_count_key),
                    )
                else:
                    st.button(
                        f"👎 {st.session_state.dislikes_clicked[dislike_count_key]}",
                        key=f"dislike_button_{ids}",
                        disabled=True 
                    )
            
            with col23: 
                
                share_key = f"share_clicked_{ids}"  
                share_count_key = f"share_count_{ids}"

                if share_key not in st.session_state:
                    st.session_state[share_key] = False  

                if share_count_key not in st.session_state.shares_clicked:
                    st.session_state.shares_clicked[share_count_key] = 0 

                def handle_share_button(share_key, share_count_key):
                    st.session_state[share_key] = True
                    st.session_state.shares_clicked[share_count_key] += 1

                if not st.session_state[share_key]:
                    st.button(
                        f"🔄 {st.session_state.shares_clicked[share_count_key]}",
                        key=f"share_button_{ids}",
                        on_click=handle_share_button,
                        args=(share_key, share_count_key),
                    )
                else:
                    st.button(
                        f"🔄 {st.session_state.shares_clicked[share_count_key]}",
                        key=f"share_button_{ids}",
                        disabled=True 
                    )

            with col24: 

                report_key = f"report_clicked_{ids}"  
                report_count_key = f"report_count_{ids}"

                if report_key not in st.session_state:
                    st.session_state[report_key] = False  

                if report_count_key not in st.session_state.reports_clicked:
                    st.session_state.reports_clicked[report_count_key] = 0 

                def handle_report_button(report_key, report_count_key):
                    st.session_state[report_key] = True
                    st.session_state.reports_clicked[report_count_key] += 1

                if not st.session_state[report_key]:
                    st.button(
                        f"🚨 {st.session_state.reports_clicked[report_count_key]}",
                        key=f"report_button_{ids}",
                        on_click=handle_report_button,
                        args=(report_key, report_count_key),
                    )
                else:
                    st.button(
                        f"🚨 {st.session_state.reports_clicked[report_count_key]}",
                        key=f"report_button_{ids}",
                        disabled=True 
                    )
                    
    likes_df = state_to_df(types='like')
    dislikes_df = state_to_df(types='dislike')
    shares_df = state_to_df(types='share')
    reports_df = state_to_df(types='report')

    new_interactions_df = likes_df.merge(dislikes_df, on='content_id').merge(shares_df, on='content_id').merge(reports_df, on='content_id')

    new_interactions_df['user_id'] = user
    new_interactions_melted = new_interactions_df.melt(
        id_vars=['content_id', 'user_id'],
        var_name='interaction_type',
        value_name='value')
    
    new_interactions_result = new_interactions_melted[new_interactions_melted['value'] == 1].drop(columns=['value'])
    new_interactions_result['interaction_date'] = datetime.today().strftime('%Y-%m-%d')
    new_interactions_result['interaction_id'] = random.randint(1000, 9999)

    df_cols = ['interaction_id', 'user_id', 'content_id', 'interaction_type', 'interaction_date']
    new_interactions_result = new_interactions_result[df_cols]
    print(new_interactions_result)

    updated_interactions = pd.concat(
        [int_df, new_interactions_result], 
        axis=0, 
        ignore_index=False)

    container_client.get_blob_client(int_client).upload_blob(
        updated_interactions.to_csv(index=False), overwrite=True)

Overwriting ./app/utils.py


In [ ]:
%%writefile {app_src_dir}/requirements.txt
azure-storage-blob>=12.0.0
azureml-core
azure-identity
streamlit
pandas
numpy

In [ ]:
%%writefile {app_src_dir}/startup.sh
#!/bin/bash

streamlit run app.py --server.port=8000 --server.enableCORS=false